In [0]:
clustering_data_path='development_042_silver_sandbox.demand_forecast.game_attributes_class_3_core_clustering_gold'
clustering_res_path='development_042_silver_sandbox.demand_forecast.clustering_labels_gold_core_class_3'
clustering_ml_atr_path='development_042_silver_sandbox.demand_forecast.clustering_exp_attr'
clustering_like_games_path='development_042_silver_sandbox.demand_forecast.clustering_core3_like_games'
# TRAIN / VALIDATION / TEST VARIABLES:
MAX_GAME_RELEASE_DATE="2026-04-30"
MIN_GAME_RELEASE_DATE="2019-01-01"
#FEATURE WEIGHTS
write='append'
FEATURE_GROUPS = {
    "form_factor": {
        "prefixes": ["form_factor__"],
        "weight": 1.5,
    },
    "merchandising": {
        "prefixes": ["merchandising__"],
        "weight": .0,# idk does it matter?
    },
    "product_segment": {
        "prefixes": ["product_segment__"],
        "weight": 1,# 
    },
    "mechanic_segment": {
        "prefixes": ["mechanic_segment__"],
        "weight": 1,# check with ricky 
    },
    "brand_evolution": {
        "prefixes": ["brand_evolution__"],
        "weight": 1,# its important for premium games not for core
    },

    # Game theme - bumped up since theme is a primary driver of clustering
    "theme": {
        "prefixes": ["theme__"],
        "weight": 2,
    },

    "art_style": {
        "prefixes": ["art_style__"],
        "weight": 2,
    },

    # Gameplay/game feature flags - bumped up since these mechanics
    # (jackpots, hold & spin, persistence, lines/ways, royals, etc.)
    # are a primary driver of clustering
    "game_features": {
        "prefixes": [
            "jackpots__",
            "must_hit_by__",
            "alt_progressive__",
            "hold_and_spin__",
            "nested_hns__",
            "percieved_persistence__",
            "real_persistence__",
            "linesorways__",
            "royals__",
            "cashonreels__",
        ],
        "weight":3,
    },
    "feature_count": {
        "columns": ["total_features"],
        "weight": 1.5,
        "numeric": True,
    },
    "persistence_score": {
        "columns": ["pp_score"],
        "weight": 1.5,# ask ricky 
        "numeric": True,
    },
    'cabinet_age':{
        'columns':['months_since_cabinet_release__log1p'],
         "weight": 1,# ask ricky 
        "numeric": True,

    }
    ,
    "bet_denomination": {
        "columns": ["ctc"],
        "weight": 1,# was 0.3
        "numeric": True,
    },

    "release_age": {
        "columns": ["months_since_release__log1p"],
        "weight": 0.5,
        "numeric": True,
    },
}


In [0]:
from pyspark.sql.functions import months_between, current_date, col

df=spark.read.table(clustering_data_path)

df = df.withColumn(
    "months_since_release",
    months_between(current_date(), col("min_game_release_date"))
)

df=df[df.product_segment=='core']
df=df[df.game_classification=='CLASS 3']
display(df)


In [0]:
# cleaning functions, fixes names, changes categorical features to one hot encoding and applies transformations
import hashlib
import re

from pyspark.sql import DataFrame
from pyspark.sql import functions as F



def _safe_feature_name(value: object) -> str:
    """
    Convert a category value into a Spark-safe column-name component.
    """
    text = str(value).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = text.strip("_")

    return text or "empty"


def _named_one_hot_encode(
    df: DataFrame,
    column: str,
    *,
    missing_label: str = "missing",
    max_categories: int = 100,
) -> tuple[DataFrame, list[str]]:
    """
    Create readable one-hot columns such as:

        form_factor__portrait

    Returns the transformed DataFrame and the generated feature names.
    """

    normalized_value = (
        F.when(
            F.col(column).isNull()
            | (F.trim(F.col(column).cast("string")) == ""),
            F.lit(missing_label),
        )
        .otherwise(F.trim(F.col(column).cast("string")))
    )

    categories = [
        row["category"]
        for row in (
            df.select(normalized_value.alias("category"))
            .distinct()
            .orderBy("category")
            .collect()
        )
    ]

    if len(categories) > max_categories:
        raise ValueError(
            f"Column '{column}' has {len(categories)} categories. "
            f"The configured maximum is {max_categories}."
        )

    generated_columns = []
    used_names = {}

    for category in categories:
        safe_category = _safe_feature_name(category)
        base_name = f"{column}__{safe_category}"
        output_name = base_name

        # Protect against collisions such as "A-B" and "A B"
        if output_name in used_names and used_names[output_name] != category:
            category_hash = hashlib.md5(
                str(category).encode("utf-8")
            ).hexdigest()[:6]

            output_name = f"{base_name}_{category_hash}"

        used_names[output_name] = category

        df = df.withColumn(
            output_name,
            F.when(
                normalized_value == F.lit(category),
                F.lit(1.0),
            ).otherwise(F.lit(0.0)),
        )

        generated_columns.append(output_name)

    return df, generated_columns


def apply_feature_transformations(
    df: DataFrame,
    transformations: dict[str, list[str]],
    *,
    drop_original_columns: bool = True,
    max_categories: int = 100,
    return_feature_columns: bool = False,
):
    """
    Apply configured feature transformations.

    Supported transformations:
        - one_hot
        - log1p
        - drop

    log1p creates:
        months_since_release__log1p

    By default, returns only the transformed Spark DataFrame.

    Set return_feature_columns=True to return:
        transformed_df, generated_feature_columns
    """

    transformed_df = df
    generated_feature_columns = []
    columns_to_drop = set()

    for column, column_transformations in transformations.items():

        if column not in transformed_df.columns:
            raise ValueError(f"Column '{column}' does not exist in the DataFrame.")

        for transformation in column_transformations:

            if transformation == "one_hot":
                transformed_df, new_columns = _named_one_hot_encode(
                    transformed_df,
                    column,
                    max_categories=max_categories,
                )

                generated_feature_columns.extend(new_columns)

                if drop_original_columns:
                    columns_to_drop.add(column)

            elif transformation == "log1p":
                output_column = f"{column}__log1p"
                # try_cast tolerates malformed numeric strings (e.g. "n/a")
                # by returning null instead of raising under ANSI mode
                numeric_value = F.col(column).try_cast("double")

                transformed_df = transformed_df.withColumn(
                    output_column,
                    F.when(
                        numeric_value >= 0,
                        F.log(numeric_value + F.lit(1.0)),
                    ).otherwise(F.lit(None).cast("double")),
                )

                generated_feature_columns.append(output_column)

                if drop_original_columns:
                    columns_to_drop.add(column)

            elif transformation == "drop":
                columns_to_drop.add(column)

            else:
                raise ValueError(
                    f"Unsupported transformation '{transformation}' "
                    f"for column '{column}'."
                )

    existing_columns_to_drop = [
        column
        for column in columns_to_drop
        if column in transformed_df.columns
    ]

    if existing_columns_to_drop:
        transformed_df = transformed_df.drop(*existing_columns_to_drop)

    if return_feature_columns:
        return transformed_df, generated_feature_columns

    return transformed_df

In [0]:
TRANSFORMATIONS = {
    "form_factor": ["one_hot"],
    "merchandising": ["one_hot"],
    "product_segment": ["one_hot"],
    "mechanic_segment": ["one_hot"],
    "brand_evolution": ["one_hot"],
    "theme": ["one_hot"],

    # Use the derived numeric age instead of one-hot encoding the date
    #"ep_min_release_date": ["drop"],

    "months_since_release": ["log1p"],
    "months_since_cabinet_release":['log1p'],
    # Categorical game attributes
    "vendor": ["drop"],
    "jackpots": ["one_hot"],
    "must_hit_by": ["one_hot"],
    "alt_progressive": ["one_hot"],
    "art_style": ["one_hot"],
    "hold_and_spin": ["one_hot"],
    "nested_hns": ["one_hot"],
    "cash_collect": ["drop"],
    "helper_character": ["drop"],
    "percieved_persistence": ["one_hot"],
    "real_persistence": ["one_hot"],
    "linesorways": ["one_hot"],
    "royals": ["one_hot"],
    "cashonreels": ["one_hot"],
    "frames": ["drop"],
    "bg_music": ["drop"],
    "visible_wheel": ["drop"],

    # Numeric-like game attributes (stored as string/long counts or scores)
    #"ctc": ["log1p"],
    # total_features: kept as-is (small values, no log1p needed); min-max scaled in FEATURE_GROUPS
    #"pp_score": kept as is small value

    # Identifiers, free-text notes, and BP/EP fuzzy-matching diagnostics -
    # not real game attributes, so they are dropped rather than encoded.
    # game_name is intentionally NOT dropped/encoded here - it's kept as
    # the join key used later to reattach dataset_split to cluster_df.
    #"cabinet": ["drop"],  # 313/315 rows are null
    "key": ["drop"],
    "helper_char_notes": ["drop"],
    #"bp_source_row": ["drop"],
    "bp_theme_name": ["drop"],
    "ep_theme_name": ["drop"],
    "correction": ["drop"],
    "theme_name": ["drop"],

}

cluster_df, clustering_feature_columns = apply_feature_transformations(
    df,
    TRANSFORMATIONS,
    drop_original_columns=True,
    return_feature_columns=True,
)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def add_train_validation_test_label(
    df: DataFrame,
    date_col: str = "min_game_release_date",
    output_col: str = "dataset_split",
    test_percent: int = 10,
    val_percent: int = 10,
) -> DataFrame:
    """
    Chronologically split rows into train, validation, and test sets.

    Example:
        test_percent=10 -> newest 10% are test
        val_percent=10  -> next-newest 10% are valid
        remaining 80%   -> train
    """

    if not 0 < test_percent < 100:
        raise ValueError("test_percent must be between 0 and 100.")

    if not 0 <= val_percent < 100:
        raise ValueError("val_percent must be between 0 and 100.")

    if test_percent + val_percent >= 100:
        raise ValueError(
            "test_percent + val_percent must be less than 100."
        )

    if date_col not in df.columns:
        raise ValueError(f"Column '{date_col}' does not exist.")

    # Newest games come first.
    order_window = Window.orderBy(
        F.col(date_col).desc_nulls_last()
    )

    # Used to calculate the total number of rows.
    total_window = Window.partitionBy()

    return (
        df
        .withColumn(
            "_split_row_number",
            F.row_number().over(order_window),
        )
        .withColumn(
            "_total_rows",
            F.count(F.lit(1)).over(total_window),
        )
        .withColumn(
            "_test_end",
            F.ceil(
                F.col("_total_rows") *
                F.lit(test_percent / 100.0)
            ),
        )
        .withColumn(
            "_valid_end",
            F.ceil(
                F.col("_total_rows") *
                F.lit((test_percent + val_percent) / 100.0)
            ),
        )
        .withColumn(
            output_col,
            F.when(
                F.col("_split_row_number") <= F.col("_test_end"),
                F.lit("test"),
            )
            .when(
                F.col("_split_row_number") <= F.col("_valid_end"),
                F.lit("valid"),
            )
            .otherwise(F.lit("train")),
        )
        .drop(
            "_split_row_number",
            "_total_rows",
            "_test_end",
            "_valid_end",
        )
    )
# Keep games inside the eligible date range first.
df = df.filter(
    F.col("min_game_release_date").between(
        F.to_date(F.lit(MIN_GAME_RELEASE_DATE)),
        F.to_date(F.lit(MAX_GAME_RELEASE_DATE)),
    )
)
df = add_train_validation_test_label(
    df,
    date_col="min_game_release_date",
    output_col="dataset_split",
    test_percent=10,
)

# Test remains held out; validation drives like-game evaluation and diagnostics.
# Training rows remain the only source of candidate like games.


df.groupBy("dataset_split").count().show()
df.groupBy("dataset_split").agg(
    F.count("*").alias("rows"),
    F.min("months_since_release").alias("earliest_release"),
    F.max("months_since_release").alias("latest_release"),
).show()

In [0]:
import math

import numpy as np
from pyspark.ml.feature import Normalizer, VectorAssembler
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def _resolve_group_columns(
    df: DataFrame,
    group_config: dict,
) -> list[str]:
    selected_columns = list(group_config.get("columns", []))

    for prefix in group_config.get("prefixes", []):
        selected_columns.extend(
            column
            for column in df.columns
            if column.startswith(prefix)
        )

    # Preserve order and remove duplicates
    return list(dict.fromkeys(selected_columns))


def prepare_weighted_training_matrix(
    df: DataFrame,
    feature_groups: dict,
    *,
    split_col: str = "dataset_split",
    train_label: str = "train",
    output_col: str = "cosine_features",
    numeric_stats: dict | None = None,
):
    """
    Prepare a weighted, L2-normalized feature matrix from filtered rows.

    Args:
        numeric_stats: Optional dict of {column_name: (min, max)}.
            If provided, uses these stats for min-max normalization
            instead of computing from the filtered data. Use this
            when preparing validation or other held-out data with
            training statistics.

    Returns:
        prepared_spark_df
        numpy_feature_matrix
        weighted_feature_columns
    """

    train_df = df.filter(F.col(split_col) == train_label)

    weighted_columns = []

    for group_name, config in feature_groups.items():
        group_columns = _resolve_group_columns(train_df, config)

        if not group_columns:
            raise ValueError(
                f"No columns found for feature group '{group_name}'."
            )

        weight = float(config.get("weight", 1.0))

        if weight < 0:
            raise ValueError(
                f"Weight for '{group_name}' cannot be negative."
            )

        scale = math.sqrt(weight)
        is_numeric = config.get("numeric", False)

        for source_column in group_columns:
            weighted_column = f"_weighted__{source_column}"

            if is_numeric:
                if numeric_stats and source_column in numeric_stats:
                    minimum, maximum = numeric_stats[source_column]
                else:
                    stats = (
                        train_df
                        .agg(
                            F.min(source_column).alias("minimum"),
                            F.max(source_column).alias("maximum"),
                        )
                        .first()
                    )
                    minimum = stats["minimum"]
                    maximum = stats["maximum"]

                if minimum is None or maximum is None:
                    raise ValueError(
                        f"Numeric feature '{source_column}' contains no "
                        "non-null training values."
                    )

                if maximum == minimum:
                    normalized_value = F.lit(0.0)
                else: #standardizing the value
                    normalized_value = (
                        F.coalesce(
                            F.col(source_column).cast("double"),
                            F.lit(float(minimum)),
                        )
                        - F.lit(float(minimum))
                    ) / F.lit(float(maximum - minimum))

                train_df = train_df.withColumn(
                    weighted_column,
                    normalized_value * F.lit(scale),
                )

            else:
                train_df = train_df.withColumn(
                    weighted_column,
                    F.coalesce(
                        F.col(source_column).cast("double"),
                        F.lit(0.0),
                    )
                    * F.lit(scale),
                )

            weighted_columns.append(weighted_column)

    assembler = VectorAssembler(
        inputCols=weighted_columns,
        outputCol="_weighted_features",
        handleInvalid="error",
    )

    normalizer = Normalizer(
        inputCol="_weighted_features",
        outputCol=output_col,
        p=2.0,
    )

    prepared_df = normalizer.transform(
        assembler.transform(train_df)
    )

    feature_rows = prepared_df.select(output_col).collect()

    matrix = np.vstack([
        row[output_col].toArray()
        for row in feature_rows
    ])

    zero_vector_mask = np.linalg.norm(matrix, axis=1) == 0

    if zero_vector_mask.any():
        raise ValueError(
            f"{zero_vector_mask.sum()} training rows have no usable "
            "feature values. Cosine distance is undefined for zero vectors."
        )

    return prepared_df, matrix, weighted_columns



In [0]:
cluster_df = cluster_df.drop("dataset_split").join(
    df.select("game_name", "dataset_split"),
    on="game_name",
    how="inner",
)

# Cast total_features and pp_score from string to double (clean up)
cluster_df = cluster_df.withColumn("total_features", F.col("total_features").try_cast("double"))
cluster_df = cluster_df.withColumn("pp_score", F.col("pp_score").try_cast("double"))

# Override: move any columns with "missing" in their name to a weight-0 group
_missing_columns = []
for _grp, _cfg in list(FEATURE_GROUPS.items()):
    _prefixes = _cfg.get("prefixes", [])
    if _prefixes:
        _group_cols = [
            c for p in _prefixes for c in cluster_df.columns if c.startswith(p)
        ]
        _missing = [c for c in _group_cols if "missing" in c.lower()]
        if _missing:
            _non_missing = [c for c in _group_cols if "missing" not in c.lower()]
            _missing_columns.extend(_missing)
            FEATURE_GROUPS[_grp] = {**_cfg, "prefixes": [], "columns": _non_missing}

if _missing_columns:
    FEATURE_GROUPS["missing_indicators"] = {
        "columns": _missing_columns,
        "weight": 0,
    }
    print(f"Masked {len(_missing_columns)} 'missing' columns with weight 0.")

prepared_train_df, X_train, weighted_feature_columns = (
    prepare_weighted_training_matrix(
        cluster_df,
        FEATURE_GROUPS,
        split_col="dataset_split",
    )
)

print("Training rows:", X_train.shape[0])
print("Feature columns:", X_train.shape[1])

In [0]:


# EVAL CLUSTERING FUNCTIOn
import pandas as pd
from scipy.cluster.hierarchy import cut_tree, linkage
from sklearn.metrics import silhouette_score


def evaluate_agglomerative_k(
    X: np.ndarray,
    *,
    k_values=range(2, 21),
    linkage_method: str = "average",
    minimum_useful_cluster_size: int = 5,
):
    """
    Build one agglomerative hierarchy and evaluate different cuts.

    Higher silhouette is better, but cluster balance and interpretability
    should also be considered.
    """

    allowed_linkages = {"average", "complete", "single"}

    if linkage_method not in allowed_linkages:
        raise ValueError(
            f"For cosine distance, choose one of {allowed_linkages}."
        )

    number_of_rows = len(X)

    if number_of_rows < 3:
        raise ValueError("At least three training rows are required.")

    hierarchy = linkage(
        X,
        method=linkage_method,
        metric="cosine",
        optimal_ordering=True,
    )

    results = []

    for k in k_values:
        if k < 2 or k >= number_of_rows:
            continue

        labels = cut_tree(
            hierarchy,
            n_clusters=[k],
        ).reshape(-1)

        cluster_sizes = np.bincount(labels)

        silhouette = silhouette_score(
            X,
            labels,
            metric="cosine",
        )

        results.append({
            "k": k,
            "silhouette": silhouette,
            "smallest_cluster": int(cluster_sizes.min()),
            "largest_cluster": int(cluster_sizes.max()),
            "largest_cluster_pct": (
                cluster_sizes.max() / number_of_rows
            ),
            "clusters_below_minimum": int(
                (cluster_sizes < minimum_useful_cluster_size).sum()
            ),
        })

    results_df = (
        pd.DataFrame(results)
        .sort_values("k")
        .reset_index(drop=True)
    )

    return results_df, hierarchy

In [0]:
k_results, hierarchy = evaluate_agglomerative_k(
    X_train,
    k_values=range(2, 21),
    linkage_method="complete",
    minimum_useful_cluster_size=5,
)

display(k_results)
import matplotlib.pyplot as plt


plt.figure(figsize=(10, 5))

plt.plot(
    k_results["k"],
    k_results["silhouette"],
    marker="o",
)

plt.xticks(k_results["k"])
plt.xlabel("Number of clusters")
plt.ylabel("Cosine silhouette score")
plt.title("Agglomerative clustering: selecting k")
plt.grid(alpha=0.3)
plt.show()
display(
    k_results.sort_values(
        ["silhouette", "clusters_below_minimum"],
        ascending=[False, True],
    )
)

In [0]:
import os
import tempfile
from datetime import datetime

import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import cut_tree, dendrogram
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
from pyspark.sql import functions as F


# ─── Training metrics ─────────────────────────────────────────────────────────

def cut_and_score(hierarchy, X_train, k):
    """Cut hierarchy at k clusters and compute training silhouette metrics."""
    labels = cut_tree(hierarchy, n_clusters=[k]).reshape(-1)
    sil_score = silhouette_score(X_train, labels, metric="cosine")
    sil_samples_arr = silhouette_samples(X_train, labels, metric="cosine")
    cluster_sizes = np.bincount(labels)
    per_cluster_sil = {
        f"silhouette_cluster_{i}": float(sil_samples_arr[labels == i].mean())
        for i in range(k)
    }
    return labels, sil_score, sil_samples_arr, cluster_sizes, per_cluster_sil


def compute_centroids(X_train, labels, k):
    """Compute cluster centroids from training data."""
    return np.vstack([X_train[labels == i].mean(axis=0) for i in range(k)])


# ─── Held-out split preparation & assignment ────────────────────────────────────────

def prepare_split_matrix(cluster_df, feature_groups, split_label):
    """Prepare valid/test rows using normalization statistics from training only."""
    if split_label not in {"valid", "test"}:
        raise ValueError("split_label must be either 'valid' or 'test'.")

    training_df = cluster_df.filter(F.col("dataset_split") == "train")
    numeric_columns = list(dict.fromkeys(
        column
        for config in feature_groups.values()
        if config.get("numeric", False)
        for column in _resolve_group_columns(training_df, config)
    ))
    stat_expressions = []
    for index, column in enumerate(numeric_columns):
        stat_expressions.extend([
            F.min(F.col(column)).alias(f"minimum_{index}"),
            F.max(F.col(column)).alias(f"maximum_{index}"),
        ])
    stats_row = training_df.agg(*stat_expressions).first()
    numeric_stats = {
        column: (stats_row[f"minimum_{index}"], stats_row[f"maximum_{index}"])
        for index, column in enumerate(numeric_columns)
    }
    missing_stats = [
        column for column, bounds in numeric_stats.items() if None in bounds
    ]
    if missing_stats:
        raise ValueError(f"Training data has no values for: {missing_stats}")

    prepared, X_split, _ = prepare_weighted_training_matrix(
        cluster_df,
        feature_groups,
        split_col="dataset_split",
        train_label=split_label,
        numeric_stats=numeric_stats,
    )
    return X_split, prepared


def assign_split_labels(X_split, centroids):
    """Assign held-out rows to their nearest training centroid."""
    distances = cdist(X_split, centroids, metric="cosine")
    return distances.argmin(axis=1), distances


def compute_validation_silhouette(X_train, X_validation, labels, validation_labels):
    """Compute silhouette for validation rows relative to the combined train/validation dataset."""
    combined_X = np.vstack([X_train, X_validation])
    combined_labels = np.concatenate([labels, validation_labels])
    combined_sil = silhouette_samples(combined_X, combined_labels, metric="cosine")
    return combined_sil[len(labels):]


# ─── Assignments DataFrame ────────────────────────────────────────────────────

def build_assignments(
    train_games_pd,
    validation_games_pd,
    test_games_pd,
    labels,
    validation_labels,
    test_labels,
    sil_samples_arr,
    validation_silhouette,
    k,
):
    """Combine train, validation, and test cluster assignments."""
    train_df = pd.DataFrame({
        "game_name": train_games_pd["game_name"],
        "cluster": labels,
        "silhouette": sil_samples_arr,
        "dataset_split": "train",
        "assignment_method": "hierarchy_cut",
    })
    validation_df = pd.DataFrame({
        "game_name": validation_games_pd["game_name"],
        "cluster": validation_labels,
        "silhouette": validation_silhouette,
        "dataset_split": "valid",
        "assignment_method": "centroid_nearest",
    })
    test_df = pd.DataFrame({
        "game_name": test_games_pd["game_name"],
        "cluster": test_labels,
        "silhouette": np.nan,
        "dataset_split": "test",
        "assignment_method": "centroid_nearest",
    })
    all_assignments = pd.concat(
        [train_df, validation_df, test_df],
        ignore_index=True,
    )
    all_assignments["k"] = k
    all_assignments["assigned_at"] = datetime.utcnow()
    all_assignments["mlflow_run_id"] = None
    return all_assignments


# --- Nearest neighbor analysis ---

def find_nearest_neighbors(
    X_target,
    X_train,
    target_labels,
    labels,
    target_games_list,
    train_games_list,
    raw_attributes_pd,
    dataset_split,
    top_n=3,
):
    """Find training like games for every valid/test target game."""
    if dataset_split not in {"valid", "test"}:
        raise ValueError("dataset_split must be either 'valid' or 'test'.")

    raw_cols = [c for c in raw_attributes_pd.columns if c != "dataset_split"]
    all_dists = cdist(X_target, X_train, metric="cosine")
    records = []

    for i, target_game in enumerate(target_games_list):
        target_raw = (
            raw_attributes_pd.loc[
                raw_attributes_pd["game_name"] == target_game,
                raw_cols,
            ]
            .iloc[0]
            .to_dict()
        )
        sorted_indices = np.argsort(all_dists[i])[:top_n]

        for rank, train_idx in enumerate(sorted_indices, start=1):
            train_game = train_games_list[train_idx]
            train_raw = (
                raw_attributes_pd.loc[
                    raw_attributes_pd["game_name"] == train_game,
                    raw_cols,
                ]
                .iloc[0]
                .to_dict()
            )
            record = {
                "target_game_name": target_game,
                "target_cluster": int(target_labels[i]),
                "dataset_split": dataset_split,
                "neighbor_rank": rank,
                "game_name": train_game,
                "train_cluster": int(labels[train_idx]),
                "cosine_distance": float(all_dists[i, train_idx]),
            }
            for col_name in raw_cols:
                record[f"target__{col_name}"] = target_raw[col_name]
                record[f"train__{col_name}"] = train_raw[col_name]
            records.append(record)

    return pd.DataFrame(records)


# --- Artifact generation ---

def _plot_dendrogram(hierarchy, k, tmp_dir, labels=None):
    """Plot full dendrogram with k cluster colors and optional game-name labels."""
    # Color threshold at the merge distance that would reduce k → k-1 clusters
    color_threshold = hierarchy[-(k - 1), 2]
    n_leaves = len(labels) if labels else hierarchy.shape[0] + 1
    fig_width = max(16, n_leaves * 0.18) if labels else 14

    fig, ax = plt.subplots(figsize=(fig_width, 10 if labels else 6))
    dendrogram(
        hierarchy,
        truncate_mode=None if labels else "level",
        p=5,
        labels=labels,
        leaf_rotation=90 if labels else 0,
        leaf_font_size=5.5 if labels else 8,
        show_leaf_counts=not bool(labels),
        color_threshold=color_threshold,
        ax=ax,
    )
    ax.axhline(color_threshold, ls="--", color="gray", alpha=0.6,
               label=f"k={k} cut (d={color_threshold:.3f})")
    ax.legend(loc="upper right", fontsize=10)
    ax.set_title(f"Full Dendrogram — {n_leaves} games, k={k} clusters (complete linkage)", fontsize=13)
    ax.set_ylabel("Cosine distance")
    path = os.path.join(tmp_dir, "dendrogram.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def _plot_cluster_sizes(cluster_sizes, k, tmp_dir):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(k), cluster_sizes, color="steelblue")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Number of games")
    ax.set_title(f"Cluster sizes (k={k})")
    ax.set_xticks(range(k))
    path = os.path.join(tmp_dir, "cluster_sizes.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def _plot_silhouette(per_cluster_sil, sil_score, k, tmp_dir):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(k), list(per_cluster_sil.values()), color="darkorange")
    ax.axhline(sil_score, ls="--", color="gray", label=f"Overall: {sil_score:.4f}")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Mean silhouette")
    ax.set_title(f"Per-cluster silhouette (k={k})")
    ax.set_xticks(range(k))
    ax.legend()
    path = os.path.join(tmp_dir, "silhouette_per_cluster.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def _plot_feature_heatmap(z_filtered, k, tmp_dir):
    """Plot non-theme feature z-score heatmap."""
    non_theme = [f for f in z_filtered.columns if not f.startswith("_weighted__theme__")]
    top_features = set()
    for i in range(k):
        row = z_filtered.iloc[i].dropna().sort_values(ascending=False)
        top_features.update(row.head(4).index.tolist())
    non_theme = sorted(set(non_theme) & top_features)
    if not non_theme:
        return None

    heatmap_data = z_filtered[non_theme].copy()
    heatmap_data.index = [f"Cluster {i}" for i in range(k)]
    fig, ax = plt.subplots(figsize=(max(12, len(non_theme) * 0.6), k * 0.5 + 2))
    sns.heatmap(heatmap_data, cmap="RdBu_r", center=0, annot=True, fmt=".1f",
                linewidths=0.5, ax=ax, cbar_kws={"label": "Z-score"})
    ax.set_title(f"Cluster Feature Profile (z-scored centroids, k={k})")
    ax.set_ylabel("")
    path = os.path.join(tmp_dir, "cluster_feature_profile_heatmap.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def _plot_theme_heatmap(z_filtered, k, tmp_dir):
    """Plot theme feature z-score heatmap (only significant themes)."""
    theme_cols = [f for f in z_filtered.columns if f.startswith("_weighted__theme__")]
    if not theme_cols:
        return None
    theme_z = z_filtered[theme_cols]
    significant = theme_z.columns[theme_z.abs().max(axis=0) > 1.0].tolist()
    if not significant:
        return None

    hm = theme_z[significant].copy()
    hm.index = [f"Cluster {i}" for i in range(k)]
    hm.columns = [c.replace("_weighted__theme__", "") for c in hm.columns]
    fig, ax = plt.subplots(figsize=(max(12, len(significant) * 0.7), k * 0.5 + 2))
    sns.heatmap(hm, cmap="RdBu_r", center=0, annot=True, fmt=".1f",
                linewidths=0.5, ax=ax, cbar_kws={"label": "Z-score"})
    ax.set_title(f"Cluster Theme Profile (z-scored, k={k})")
    ax.set_ylabel("")
    path = os.path.join(tmp_dir, "cluster_theme_profile_heatmap.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def _plot_similarity_heatmap(centroids, k, tmp_dir):
    sim_matrix = 1 - cdist(centroids, centroids, metric="cosine")
    sim_df = pd.DataFrame(
        sim_matrix,
        index=[f"Cluster {i}" for i in range(k)],
        columns=[f"Cluster {i}" for i in range(k)],
    )
    fig, ax = plt.subplots(figsize=(k * 0.8 + 2, k * 0.7 + 2))
    sns.heatmap(sim_df, cmap="YlOrRd", annot=True, fmt=".2f",
                linewidths=0.5, ax=ax, vmin=0, vmax=1,
                cbar_kws={"label": "Cosine similarity"})
    ax.set_title(f"Inter-cluster Cosine Similarity (k={k})")
    path = os.path.join(tmp_dir, "cluster_similarity_heatmap.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def _plot_pca_map(centroids, cluster_sizes, k, tmp_dir):
    pca = PCA(n_components=2)
    coords = pca.fit_transform(centroids)
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.scatter(coords[:, 0], coords[:, 1], s=cluster_sizes * 3,
              c=range(k), cmap="tab10", edgecolors="black", linewidths=0.8, alpha=0.85)
    for i in range(k):
        ax.annotate(f"C{i}\n(n={cluster_sizes[i]})", (coords[i, 0], coords[i, 1]),
                    fontsize=8, ha="center", va="bottom")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
    ax.set_title(f"Cluster Centroid PCA Map (k={k})")
    ax.grid(alpha=0.3)
    path = os.path.join(tmp_dir, "cluster_centroid_pca_map.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def compute_z_scored_profiles(centroids_pd, k, raw_threshold=0.05):
    """Compute z-scored centroid profiles and top features per cluster."""
    features = centroids_pd.drop(columns=["cluster"])
    means = features.mean(axis=0)
    stds = features.std(axis=0).replace(0, np.nan)
    z = (features - means) / stds
    z_filtered = z.where(features.abs() >= raw_threshold)

    top_features_per_cluster = {}
    for i in range(k):
        row = z_filtered.iloc[i].dropna().sort_values(ascending=False)
        top_features_per_cluster[i] = row.head(4)

    return z_filtered, top_features_per_cluster


# ─── Main orchestrator ────────────────────────────────────────────────────────

def log_clustering_run(
    hierarchy, X_train, cluster_df, prepared_train_df, df,
    feature_groups, weighted_feature_columns, k_results,
    clustering_data_path,
    k=6, top_n_neighbors=3,
    experiment_name="/Users/soheil.latifi2@igt.com/DemandForeCast/clustering_experiment",
):
    """
    End-to-end clustering assignment, evaluation, and MLflow logging.

    Returns current-run assignments, like games, and experiment attributes.
    """
    mlflow.set_experiment(experiment_name)

    # 1. Training cut & metrics
    labels, sil_score, sil_samples_arr, cluster_sizes, per_cluster_sil = (
        cut_and_score(hierarchy, X_train, k)
    )
    centroids = compute_centroids(X_train, labels, k)

    # 2. Prepare both held-out splits with training-only normalization
    X_validation, validation_prepared = prepare_split_matrix(
        cluster_df, feature_groups, "valid",
    )
    X_test, test_prepared = prepare_split_matrix(
        cluster_df, feature_groups, "test",
    )
    validation_labels, _ = assign_split_labels(X_validation, centroids)
    test_labels, _ = assign_split_labels(X_test, centroids)
    validation_silhouette = compute_validation_silhouette(
        X_train, X_validation, labels, validation_labels,
    )

    # 3. Build assignments. Test silhouette stays unset to avoid test tuning.
    train_games_pd = prepared_train_df.select("game_name").toPandas()
    validation_games_pd = validation_prepared.select("game_name").toPandas()
    test_games_pd = test_prepared.select("game_name").toPandas()
    all_assignments = build_assignments(
        train_games_pd,
        validation_games_pd,
        test_games_pd,
        labels,
        validation_labels,
        test_labels,
        sil_samples_arr,
        validation_silhouette,
        k,
    )

    # 4. Build one like-game result for valid and test targets
    raw_attributes_pd = df.toPandas()
    validation_like_games = find_nearest_neighbors(
        X_validation,
        X_train,
        validation_labels,
        labels,
        validation_games_pd["game_name"].tolist(),
        train_games_pd["game_name"].tolist(),
        raw_attributes_pd,
        dataset_split="valid",
        top_n=top_n_neighbors,
    )
    test_like_games = find_nearest_neighbors(
        X_test,
        X_train,
        test_labels,
        labels,
        test_games_pd["game_name"].tolist(),
        train_games_pd["game_name"].tolist(),
        raw_attributes_pd,
        dataset_split="test",
        top_n=top_n_neighbors,
    )
    like_games_result = pd.concat(
        [validation_like_games, test_like_games],
        ignore_index=True,
    )

    # 5. Centroids & z-scores
    centroids_pd = pd.DataFrame(centroids, columns=weighted_feature_columns)
    centroids_pd.insert(0, "cluster", range(k))
    z_filtered, top_features_per_cluster = compute_z_scored_profiles(centroids_pd, k)

    # 6. MLflow run
    with mlflow.start_run(run_name=f"agglomerative_k{k}") as run:
        all_assignments["mlflow_run_id"] = run.info.run_id
        like_games_result["mlflow_run_id"] = run.info.run_id
        like_games_result["experiment_time"] = datetime.utcnow()

        attr_records = []
        for gname, gcfg in feature_groups.items():
            group_columns = list(gcfg.get("columns", []))
            for prefix in gcfg.get("prefixes", []):
                group_columns.extend(
                    c for c in cluster_df.columns if c.startswith(prefix)
                )
            attr_records.append({
                "feature_group": gname,
                "features": ",".join(dict.fromkeys(group_columns)),
                "weight": float(gcfg.get("weight", 1.0)),
                "is_numeric": bool(gcfg.get("numeric", False)),
                "run_id": run.info.run_id,
                "max_clusters": k,
                "similarity_method": "cosine",
                "linkage_type": "complete",
                "algorithm": "agglomerative",
                "clustering_group": "core_class_3",
            })
        experiment_attributes = pd.DataFrame(attr_records)

        # Parameters
        mlflow.log_param("k", k)
        mlflow.log_param("linkage_method", "complete")
        mlflow.log_param("distance_metric", "cosine")
        mlflow.log_param("normalization", "L2")
        mlflow.log_param("evaluation_split", "valid")
        mlflow.log_param("training_rows", X_train.shape[0])
        mlflow.log_param("validation_rows", X_validation.shape[0])
        mlflow.log_param("test_rows", X_test.shape[0])
        mlflow.log_param("feature_dimensions", X_train.shape[1])
        for gname, gcfg in feature_groups.items():
            mlflow.log_param(f"weight__{gname}", gcfg.get("weight", 1.0))

        # Metrics
        mlflow.log_metric("silhouette_score_train", sil_score)
        mlflow.log_metric("silhouette_score_validation", float(validation_silhouette.mean()))
        mlflow.log_metric("smallest_cluster_size", int(cluster_sizes.min()))
        mlflow.log_metric("largest_cluster_size", int(cluster_sizes.max()))
        mlflow.log_metric("largest_cluster_pct", cluster_sizes.max() / len(labels))
        mlflow.log_metric("cluster_size_std", float(cluster_sizes.std()))
        for cid, sval in per_cluster_sil.items():
            mlflow.log_metric(cid, sval)

        # Artifacts
        with tempfile.TemporaryDirectory() as tmp_dir:
            train_labels_list = train_games_pd["game_name"].tolist()
            for artifact_path in [
                _plot_dendrogram(hierarchy, k, tmp_dir, labels=train_labels_list),
                _plot_cluster_sizes(cluster_sizes, k, tmp_dir),
                _plot_silhouette(per_cluster_sil, sil_score, k, tmp_dir),
                _plot_feature_heatmap(z_filtered, k, tmp_dir),
                _plot_theme_heatmap(z_filtered, k, tmp_dir),
                _plot_similarity_heatmap(centroids, k, tmp_dir),
                _plot_pca_map(centroids, cluster_sizes, k, tmp_dir),
            ]:
                if artifact_path:
                    mlflow.log_artifact(artifact_path)

            # CSV artifacts
            for name, data in [
                ("cluster_assignments.csv", all_assignments),
                ("k_evaluation_results.csv", k_results),
                ("like_games.csv", like_games_result),
                ("cluster_centroids.csv", centroids_pd),
                ("clustering_attributes.csv", experiment_attributes),
            ]:
                csv_path = os.path.join(tmp_dir, name)
                data.to_csv(csv_path, index=False)
                mlflow.log_artifact(csv_path)

        # Print distinctiveness summary
        print("\n── Top distinctive features per cluster (z-scored, filtered) ──")
        for ci, feats in top_features_per_cluster.items():
            feat_str = ", ".join(
                f"{fn.replace('_weighted__', '')}={fv:.2f}" for fn, fv in feats.items()
            )
            print(f"  Cluster {ci}: {feat_str}")

        # Tags
        mlflow.set_tag("task", "clustering")
        mlflow.set_tag("algorithm", "agglomerative")
        mlflow.set_tag("data_source", clustering_data_path)

        print(f"\n✓ MLflow run logged: {run.info.run_id}")
        print(f"  Experiment: {experiment_name}")
        print(f"  k={k} | Silhouette train={sil_score:.4f} | validation={float(validation_silhouette.mean()):.4f}")
        print(f"  Cluster sizes (train): {cluster_sizes.tolist()}")
        print(f"  Assignment rows ready: {len(all_assignments)}")
        print(f"  Like-game rows ready: {len(like_games_result)}")

    return all_assignments, like_games_result, experiment_attributes

In [0]:
from scipy.cluster.hierarchy import dendrogram, linkage as hc_linkage
from pyspark.sql import functions as F

CHOSEN_K = int(dbutils.widgets.get("chosen_k"))

# --- Get train names and matrix (already available) ---
train_names = prepared_train_df.select("game_name").toPandas()["game_name"].tolist()

# --- Prepare validation matrix using training normalization stats ---
X_validation, validation_prepared = prepare_split_matrix(
    cluster_df, FEATURE_GROUPS, "valid",
)
validation_names = validation_prepared.select("game_name").toPandas()["game_name"].tolist()

# --- Combine train + validation into a single matrix and recompute hierarchy ---
X_combined = np.vstack([X_train, X_validation])
combined_hierarchy = hc_linkage(X_combined, method="complete", metric="cosine", optimal_ordering=True)

# --- Build labels: append train or validation suffix ---
combined_labels = (
    [f"{name}  [train]" for name in train_names]
    + [f"{name}  [VALIDATION]" for name in validation_names]
)

total_games = len(combined_labels)
color_threshold = combined_hierarchy[-(CHOSEN_K - 1), 2]

plt.figure(figsize=(max(18, total_games * 0.2), 11))

dendrogram(
    combined_hierarchy,
    truncate_mode=None,
    labels=combined_labels,
    leaf_rotation=90,
    leaf_font_size=5,
    color_threshold=color_threshold,
)

plt.axhline(color_threshold, ls="--", color="gray", alpha=0.6,
            label=f"k={CHOSEN_K} cut (d={color_threshold:.3f})")
plt.legend(loc="upper right", fontsize=10)
plt.ylabel("Cosine linkage distance")
plt.title(
    f"Full Dendrogram — {total_games} games ({len(train_names)} train + {len(validation_names)} validation), "
    f"k={CHOSEN_K} clusters (complete linkage)",
    fontsize=13,
)
plt.tight_layout()
plt.show()

In [0]:
# ─── Parameters ───────────────────────────────────────────────────────────────
CHOSEN_K = int(dbutils.widgets.get("chosen_k"))
TOP_N_NEIGHBORS = int(dbutils.widgets.get("top_n_neighbors"))
# ──────────────────────────────────────────────────────────────────────────────

all_assignments, like_games_result, experiment_attributes = log_clustering_run(
    hierarchy=hierarchy,
    X_train=X_train,
    cluster_df=cluster_df,
    prepared_train_df=prepared_train_df,
    df=df,
    feature_groups=FEATURE_GROUPS,
    weighted_feature_columns=weighted_feature_columns,
    k_results=k_results,
    clustering_data_path=clustering_data_path,
    k=CHOSEN_K,
    top_n_neighbors=TOP_N_NEIGHBORS,
)

display(all_assignments.sort_values(["dataset_split", "cluster", "game_name"]))
display(
    like_games_result.sort_values(
        ["dataset_split", "target_game_name", "neighbor_rank"],
    )
)

In [0]:
# Persist only the current analysis results.
expname=str(dbutils.widgets.get("experiment_name"))
assignments_sdf = spark.createDataFrame(all_assignments)
like_games_sdf = spark.createDataFrame(like_games_result)
attributes_sdf = spark.createDataFrame(experiment_attributes)
assignments_sdf = (
    spark.createDataFrame(all_assignments)
    .withColumn("experiment_name", F.lit(expname))
)

like_games_sdf = (
    spark.createDataFrame(like_games_result)
    .withColumn("experiment_name", F.lit(expname))
)

attributes_sdf = (
    spark.createDataFrame(experiment_attributes)
    .withColumn("experiment_name", F.lit(expname))
)

(
    assignments_sdf.write.mode(write)
    .option("overwriteSchema", "true")
    .saveAsTable(clustering_res_path)
)
(
    like_games_sdf.write.mode(write)
    .option("overwriteSchema", "true")
    .saveAsTable(clustering_like_games_path)
)
(
    attributes_sdf.write.mode(write)
    .option("overwriteSchema", "true")
    .saveAsTable(clustering_ml_atr_path)
)

print("Current analysis written to:")
print(f"  Assignments: {clustering_res_path}")
print(f"  Like games: {clustering_like_games_path}")
print(f"  Attributes: {clustering_ml_atr_path}")

display(
    like_games_sdf.groupBy("dataset_split").count().orderBy("dataset_split")
)
display(like_games_sdf.orderBy("dataset_split", "target_game_name", "neighbor_rank"))


In [0]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist

# Compute validation-to-training cosine distances
# (each validation game selects like games from the training set)
all_dists = cdist(X_validation, X_train, metric="cosine")


n_validation_games = all_dists.shape[0]
k_range = range(2, 11)

avg_similarities = []
for n in k_range:
    # For each game, get the average cosine similarity of its top-n neighbors
    sims_per_game = []
    for i in range(n_validation_games):
        sorted_dists = np.sort(all_dists[i])[:n]
        avg_sim = (1 - sorted_dists).mean()  # cosine similarity = 1 - distance
        sims_per_game.append(avg_sim)
    avg_similarities.append(np.mean(sims_per_game))

# Plot
plt.figure(figsize=(10, 5))
plt.plot(list(k_range), avg_similarities, marker="o", color="teal", linewidth=2)
plt.xlabel("Number of Like Games (neighbors)")
plt.ylabel("Average Cosine Similarity")
plt.title("Validation Games: Average Cosine Similarity vs Number of Like Games")
plt.xticks(list(k_range))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print the values
for n, sim in zip(k_range, avg_similarities):
    print(f"  n={n:>2d}  |  avg cosine similarity = {sim:.4f}")

In [0]:
%skip
from pyspark.sql import functions as F


# ============================================================
# Load tables
# ============================================================

assignments_df = spark.table(clustering_res_path)
like_games_df = spark.table(clustering_like_games_path)
attributes_df = spark.table(clustering_ml_atr_path)


# ============================================================
# Collect unique experiment/run pairs
#
# Assignments and like-games: mlflow_run_id
# Attributes: run_id
# ============================================================

all_unique_pairs = (
    assignments_df
    .select(
        "experiment_name",
        "mlflow_run_id",
    )
    .unionByName(
        like_games_df.select(
            "experiment_name",
            "mlflow_run_id",
        )
    )
    .unionByName(
        attributes_df.select(
            "experiment_name",
            F.col("run_id").alias("mlflow_run_id"),
        )
    )
    .filter(
        F.col("experiment_name").isNotNull()
        & F.col("mlflow_run_id").isNotNull()
    )
    .distinct()
)


# ============================================================
# Keep one run ID per experiment name
#
# Spark does not guarantee insertion order, so min() selects
# the alphabetically first run ID deterministically.
# ============================================================

valid_pairs = (
    all_unique_pairs
    .groupBy("experiment_name")
    .agg(
        F.min("mlflow_run_id").alias("mlflow_run_id")
    )
)

print("Pairs that will be retained:")

display(
    valid_pairs.orderBy("experiment_name")
)


# ============================================================
# Filter assignments
# ============================================================

assignments_clean = (
    assignments_df
    .join(
        valid_pairs,
        on=["experiment_name", "mlflow_run_id"],
        how="left_semi",
    )
    .localCheckpoint(eager=True)
)


# ============================================================
# Filter like-games
# ============================================================

like_games_clean = (
    like_games_df
    .join(
        valid_pairs,
        on=["experiment_name", "mlflow_run_id"],
        how="left_semi",
    )
    .localCheckpoint(eager=True)
)


# ============================================================
# Filter attributes
#
# Match attributes.run_id against valid_pairs.mlflow_run_id,
# while retaining the original attributes columns.
# ============================================================

attributes_clean = (
    attributes_df.alias("attributes")
    .join(
        valid_pairs.alias("valid"),
        (
            F.col("attributes.experiment_name")
            == F.col("valid.experiment_name")
        )
        & (
            F.col("attributes.run_id")
            == F.col("valid.mlflow_run_id")
        ),
        how="left_semi",
    )
    .localCheckpoint(eager=True)
)


# ============================================================
# Review counts before overwriting
# ============================================================

print(
    f"Assignments: {assignments_df.count():,} "
    f"-> {assignments_clean.count():,}"
)

print(
    f"Like games: {like_games_df.count():,} "
    f"-> {like_games_clean.count():,}"
)

print(
    f"Attributes: {attributes_df.count():,} "
    f"-> {attributes_clean.count():,}"
)


# ============================================================
# Overwrite original tables
# ============================================================

(
    assignments_clean.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(clustering_res_path)
)

(
    like_games_clean.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(clustering_like_games_path)
)

(
    attributes_clean.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(clustering_ml_atr_path)
)

print("Cleanup completed successfully.")

In [0]:
display(assignments_df) 
display(like_games_df) 
display(attributes_df )